# 03 · Incident brief
When the controller check raises an alarm, a local language model writes a short plain-English brief.
IBM Granite 3.3 2B Instruct runs on our LinuxONE server with llama.cpp, so no plant data leaves the
machine. The facts and first steps come from our code; the model only writes the explanation.
About 35 to 60 seconds per brief on 3 processors, so it runs only when an incident is raised.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

In [2]:
import json, time
from llama_cpp import Llama
from src.data_loading import PROCESSED
MODEL = ROOT / "models/granite-3.3-2b-instruct-be.Q4_K_M.gguf"
llm = Llama(model_path=str(MODEL), n_ctx=2048, n_threads=3, verbose=False)
SYSTEM = ("You write incident briefs for operators at small water utilities who are not security experts. "
          "Use only the facts given. Do not invent details. Write exactly two short sentences: "
          "what most likely happened, and why the system thinks so.")
STEPS = ["Switch the affected pump or valve to manual and check the tank level directly.",
         "Disconnect the controller from the internet and remote-access tools.",
         "Change default and shared passwords on the controller and operator screens.",
         "Compare the controller program with your last known-good copy.",
         "Report to CISA and your local FBI field office."]

In [3]:
# Build the alert from a real detection in notebook 02 (attack A214 on HAI 22.04)
traces = json.loads((PROCESSED / "controller_check_traces.json").read_text())
tr = next(t for t in traces if t["id"] == "A214")
peak = max(tr["score"])
facts = (f"Verdict: attack likely. On the Siemens-controlled water process, the valve command stopped matching what "
         f"the controller normally commands for this water level, peaking at {peak:.1f} times the alarm level. "
         f"The alarm fired {tr['seconds_to_flag']:.0f} seconds after the change began. The level reading itself stayed in range.")
t = time.time()
out = llm.create_chat_completion(messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": facts}],
                                 max_tokens=90, temperature=0.2)
print(facts, "\n")
print(out["choices"][0]["message"]["content"].strip(), "\n")
print("First steps:")
for s in STEPS: print(" -", s)
print(f"\n(generated on IBM LinuxONE in {time.time() - t:.0f} s)")

Verdict: attack likely. On the Siemens-controlled water process, the valve command stopped matching what the controller normally commands for this water level, peaking at 3.8 times the alarm level. The alarm fired 17 seconds after the change began. The level reading itself stayed in range. 

Most likely, a cyber-attack disrupted the valve command, causing it to operate outside normal parameters, which triggered the alarm 17 seconds later. The system identified this anomaly as unusual due to the valve operation deviating from its expected pattern while the water level remained within safe limits. 

First steps:
 - Switch the affected pump or valve to manual and check the tank level directly.
 - Disconnect the controller from the internet and remote-access tools.
 - Change default and shared passwords on the controller and operator screens.
 - Compare the controller program with your last known-good copy.
 - Report to CISA and your local FBI field office.

(generated on IBM LinuxONE in 6